# S4 · Regresión distribuida con Spark MLlib

**Integrante** · Leonardo Jesus Huaman Arhuata

Equipo 04 · AeroStreamPeru · Ciclo 9

**Objetivo de negocio.** Pronosticar la cantidad promedio de aeronaves simultáneamente en vuelo durante la siguiente ventana. El error queda en aeronaves y ayuda a juzgar el tamaño del cambio que convendría revisar.

## 1. Variable objetivo y variables predictoras

El objetivo es **`volumen_promedio_t1`**, el valor real de `volumen_promedio_t` en la siguiente ventana de diez minutos. Las predictoras de la lista se conocen al cerrar la ventana actual. La hora y el día de la semana corresponden a Lima. La sección 3 construye la etiqueta y la sección 4 conecta las predictoras con `VectorAssembler`.

In [37]:
INTEGRANTE = "leonardo"
INDICADOR_ACTUAL = "volumen_promedio_t"
VARIABLE_OBJETIVO = "volumen_promedio_t1"
UNIDAD = "aeronaves"

VARIABLES_PREDICTORAS = [
    "volumen_promedio_t",
    "volumen_min_t",
    "volumen_max_t",
    "velocidad_mediana_t",
    "altitud_mediana_t",
    "hora_del_dia",
    "dia_semana",
]

## 2. Fuente y calidad

**Bronze** contiene el histórico de Trino. **Silver** contiene los estados validados y deduplicados. **Gold**, producido y verificado en S3, contiene los indicadores por ventana y está particionado por `fecha_utc`. Esta práctica usa una sola fuente histórica preparada para el proyecto, el bloque del 1 al 5 de enero con cadencia de dos minutos. Comprobamos el esquema antes de entrenar.

In [38]:
from pathlib import Path

from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import Imputer, VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.types import DateType, DoubleType, IntegerType, LongType, StructType, TimestampType

RUTA_GOLD = Path("/opt/s03-procesamiento-calidad-datos/artifacts") / INTEGRANTE / "gold_ventanas"
RUTA_MODELO = Path("/opt/s04-ml-distribuido-regresion/artifacts") / INTEGRANTE / "modelo_ganador"
CAPTURAS_MINIMAS = 4
INICIO_VALIDACION = "2026-01-04"
INICIO_PRUEBA = "2026-01-05"
SEMILLA = 42

if not RUTA_GOLD.is_dir():
    raise FileNotFoundError(f"Ejecuta el notebook S3 de {INTEGRANTE} para generar {RUTA_GOLD}")

spark = (
    SparkSession.builder.master("local[2]")
    .appName(f"AeroStreamPeru-S04-{INTEGRANTE}")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "512m")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

In [39]:
esquema = (
    StructType()
    .add("inicio_utc", TimestampType())
    .add("fecha_utc", DateType())
    .add("capturas_observadas", LongType())
)
for columna in VARIABLES_PREDICTORAS:
    if columna in {"hora_del_dia", "dia_semana"}:
        tipo = IntegerType()
    elif columna in {"volumen_min_t", "volumen_max_t"}:
        tipo = LongType()
    else:
        tipo = DoubleType()
    esquema.add(columna, tipo)

tipos_gold = {campo.name: campo.dataType for campo in spark.read.parquet(str(RUTA_GOLD)).schema}
incompatibles = [campo.name for campo in esquema if tipos_gold.get(campo.name) != campo.dataType]
assert not incompatibles, f"Regenera Gold de S3. Columnas incompatibles: {incompatibles}"

gold = spark.read.schema(esquema).parquet(str(RUTA_GOLD)).cache()
assert not gold.groupBy("inicio_utc").count().where("count > 1").take(1)
print(f"Ventanas de Gold leídas: {gold.count():,}")
gold.groupBy("fecha_utc").count().orderBy("fecha_utc").show()

Ventanas de Gold leídas: 588
+----------+-----+
| fecha_utc|count|
+----------+-----+
|2026-01-01|  144|
|2026-01-02|  120|
|2026-01-03|   36|
|2026-01-04|  144|
|2026-01-05|  144|
+----------+-----+



## 3. Construcción de la etiqueta

`lead()` toma el valor posterior en orden temporal. Exigimos que avance exactamente diez minutos para que una ventana ausente no estire el horizonte. Para esta práctica se requieren al menos cuatro de las cinco capturas programadas en ambas ventanas y valores presentes del indicador actual y futuro. Esta cobertura es un criterio de la práctica.

In [40]:
orden_temporal = Window.orderBy("inicio_utc")
con_ventana_siguiente = (
    gold.withColumn("inicio_objetivo_utc", F.lead("inicio_utc").over(orden_temporal))
    .withColumn(VARIABLE_OBJETIVO, F.lead(INDICADOR_ACTUAL).over(orden_temporal))
    .withColumn("capturas_objetivo", F.lead("capturas_observadas").over(orden_temporal))
)
es_contigua = F.col("inicio_objetivo_utc") == F.col("inicio_utc") + F.expr("INTERVAL 10 MINUTES")
pares_contiguos = con_ventana_siguiente.filter(es_contigua).cache()
cobertura_suficiente = (
    (F.col("capturas_observadas") >= CAPTURAS_MINIMAS)
    & (F.col("capturas_objetivo") >= CAPTURAS_MINIMAS)
)
datos_modelo = (
    pares_contiguos.filter(cobertura_suficiente)
    .na.drop(subset=[INDICADOR_ACTUAL, VARIABLE_OBJETIVO])
    .withColumn("fecha_objetivo_utc", F.to_date("inicio_objetivo_utc"))
    .cache()
)
print(f"Ventanas con sucesora contigua: {pares_contiguos.count():,}")
print(f"Pares elegibles para modelar: {datos_modelo.count():,}")
datos_modelo.orderBy("inicio_utc").select(
    "inicio_utc", INDICADOR_ACTUAL, "inicio_objetivo_utc", VARIABLE_OBJETIVO
).show(5, truncate=False)

Ventanas con sucesora contigua: 586
Pares elegibles para modelar: 585
+-------------------+------------------+-------------------+-------------------+
|inicio_utc         |volumen_promedio_t|inicio_objetivo_utc|volumen_promedio_t1|
+-------------------+------------------+-------------------+-------------------+
|2026-01-01 00:00:00|8.5               |2026-01-01 00:10:00|7.6                |
|2026-01-01 00:10:00|7.6               |2026-01-01 00:20:00|7.4                |
|2026-01-01 00:20:00|7.4               |2026-01-01 00:30:00|9.8                |
|2026-01-01 00:30:00|9.8               |2026-01-01 00:40:00|7.2                |
|2026-01-01 00:40:00|7.2               |2026-01-01 00:50:00|5.6                |
+-------------------+------------------+-------------------+-------------------+
only showing top 5 rows


## 4. División cronológica, imputación y vector

Entrenamos con las fechas previas al 4 de enero, comparamos modelos el 4 y evaluamos al ganador el 5. Cada ejemplo tiene su ventana de origen y la ventana objetivo en el mismo periodo. Separar por fechas reproduce el uso de un modelo que predice después de entrenar.

El imputador aprende medianas de las predictoras en entrenamiento y `VectorAssembler` produce `features`. La regresión lineal estandariza internamente durante el ajuste. El bosque trabaja con las predictoras en sus escalas originales.

In [41]:
inicio_validacion = F.to_date(F.lit(INICIO_VALIDACION))
inicio_prueba = F.to_date(F.lit(INICIO_PRUEBA))

df_train = datos_modelo.filter(F.col("fecha_objetivo_utc") < inicio_validacion).cache()
df_val = datos_modelo.filter(
    (F.col("fecha_utc") >= inicio_validacion) & (F.col("fecha_objetivo_utc") < inicio_prueba)
).cache()
df_test = datos_modelo.filter(F.col("fecha_utc") >= inicio_prueba).cache()

for nombre, datos in [("entrenamiento", df_train), ("validación", df_val), ("prueba", df_test)]:
    resumen = datos.agg(
        F.count("*").alias("filas"),
        F.min("inicio_utc").alias("desde"),
        F.max("inicio_objetivo_utc").alias("hasta"),
    ).first()
    assert resumen.filas > 1, f"Periodo sin pares suficientes: {nombre}"
    print(f"{nombre}: {resumen.filas} pares · {resumen.desde} a {resumen.hasta} UTC")

columnas_imputadas = [f"{columna}_imputada" for columna in VARIABLES_PREDICTORAS]
imputador = Imputer(
    strategy="median", inputCols=VARIABLES_PREDICTORAS, outputCols=columnas_imputadas
)
ensamblador = VectorAssembler(inputCols=columnas_imputadas, outputCol="features")

entrenamiento: 297 pares · 2026-01-01 00:00:00 a 2026-01-03 23:50:00 UTC
validación: 143 pares · 2026-01-04 00:00:00 a 2026-01-04 23:50:00 UTC
prueba: 143 pares · 2026-01-05 00:00:00 a 2026-01-05 23:50:00 UTC


## 5. Modelos candidatos y comparación

Ajustamos una regresión lineal base, tres configuraciones regularizadas y un bosque aleatorio. Todos se comparan en validación con RMSE, MAE y R². RMSE penaliza más los errores grandes y decide el ganador. MAE sirve como desempate. La importancia del bosque indica qué variables usó para reducir el error al dividir los datos.

In [42]:
def medir(predicciones):
    resultados = {}
    for metrica in ["rmse", "mae", "r2"]:
        evaluador = RegressionEvaluator(
            labelCol=VARIABLE_OBJETIVO, predictionCol="prediction", metricName=metrica
        )
        resultados[metrica] = evaluador.evaluate(predicciones)
    return resultados


configuraciones_lineales = [
    ("Lineal base", 0.0, 0.0),
    ("Ridge", 0.1, 0.0),
    ("Lasso", 0.1, 1.0),
    ("Elastic Net", 0.1, 0.5),
]
candidatos = []
for nombre, reg_param, elastic_net in configuraciones_lineales:
    estimador = LinearRegression(
        labelCol=VARIABLE_OBJETIVO, featuresCol="features",
        regParam=reg_param, elasticNetParam=elastic_net,
        maxIter=80, standardization=True,
    )
    candidatos.append((nombre, estimador, reg_param, elastic_net))

bosque = RandomForestRegressor(
    labelCol=VARIABLE_OBJETIVO, featuresCol="features",
    numTrees=24, maxDepth=4, seed=SEMILLA,
)
candidatos.append(("Bosque aleatorio", bosque, None, None))

preprocesador = Pipeline(stages=[imputador, ensamblador]).fit(df_train)
train_con_vector = preprocesador.transform(df_train).select("features", VARIABLE_OBJETIVO).cache()
validacion_con_vector = preprocesador.transform(df_val).select("features", VARIABLE_OBJETIVO).cache()

modelos = {}
resultados = []
for nombre, estimador, reg_param, elastic_net in candidatos:
    modelo = estimador.fit(train_con_vector)
    modelos[nombre] = modelo
    predicciones = modelo.transform(validacion_con_vector).select(VARIABLE_OBJETIVO, "prediction").cache()
    resultados.append({
        "modelo": nombre,
        "regParam": reg_param,
        "elasticNetParam": elastic_net,
        **medir(predicciones),
    })
    predicciones.unpersist()
    print(f"Evaluado: {nombre}", flush=True)

train_con_vector.unpersist()
validacion_con_vector.unpersist()

spark.createDataFrame(resultados).select(
    "modelo", "regParam", "elasticNetParam", "rmse", "mae", "r2"
).orderBy("rmse", "mae", "modelo").show(truncate=False)

mejor = min(resultados, key=lambda fila: (fila["rmse"], fila["mae"], fila["modelo"]))
nombre_ganador = mejor["modelo"]
modelo_ganador = PipelineModel(stages=[*preprocesador.stages, modelos[nombre_ganador]])
print(f"Ganador en validación: {nombre_ganador}")

pesos = modelos["Bosque aleatorio"].featureImportances.toArray()
importancia = [(columna, float(peso)) for columna, peso in zip(VARIABLES_PREDICTORAS, pesos)]
spark.createDataFrame(importancia, ["predictora", "importancia"]).orderBy(
    F.col("importancia").desc(), "predictora"
).show(truncate=False)

Evaluado: Lineal base
Evaluado: Ridge
Evaluado: Lasso
Evaluado: Elastic Net
Evaluado: Bosque aleatorio


+----------------+--------+---------------+------------------+------------------+------------------+
|modelo          |regParam|elasticNetParam|rmse              |mae               |r2                |
+----------------+--------+---------------+------------------+------------------+------------------+
|Elastic Net     |0.1     |0.5            |1.48993232204443  |1.1921767874653062|0.6593854751761312|
|Ridge           |0.1     |0.0            |1.4947983402824732|1.1940889815915465|0.6571569940523436|
|Lasso           |0.1     |1.0            |1.4978983130802914|1.1983230290530937|0.6557335163744802|
|Lineal base     |0.0     |0.0            |1.5065685129304422|1.1994836882742057|0.6517365858295261|
|Bosque aleatorio|NULL    |NULL           |1.5932330391133098|1.2568502533583938|0.6105168372354737|
+----------------+--------+---------------+------------------+------------------+------------------+

Ganador en validación: Elastic Net
+-------------------+--------------------+
|predictora 

## 6. Evaluación de negocio en prueba

Medimos al ganador una sola vez en la prueba reservada. La persistencia usa el valor de la ventana actual como pronóstico. Esta referencia ayuda a interpretar si el modelo mejora una regla sencilla. El MAE muestra cuánto se desvía habitualmente la predicción en aeronaves. La prueba cubre un día, por lo que la capacidad de generalizar a otros periodos queda por comprobar.

In [43]:
predicciones_prueba = modelo_ganador.transform(df_test).select(
    "inicio_utc", "inicio_objetivo_utc", VARIABLE_OBJETIVO, "prediction"
).cache()
referencia_prueba = df_test.select(VARIABLE_OBJETIVO, F.col(INDICADOR_ACTUAL).alias("prediction")).cache()

metricas_ganador = medir(predicciones_prueba)
metricas_referencia = medir(referencia_prueba)
resumen_prueba = [
    (nombre_ganador, metricas_ganador["rmse"], metricas_ganador["mae"], metricas_ganador["r2"]),
    ("Persistencia", metricas_referencia["rmse"], metricas_referencia["mae"], metricas_referencia["r2"]),
]
spark.createDataFrame(resumen_prueba, ["modelo", "rmse", "mae", "r2"]).show(truncate=False)

predicciones_prueba.orderBy("inicio_objetivo_utc").withColumn(
    "error_absoluto", F.abs(F.col("prediction") - F.col(VARIABLE_OBJETIVO))
).show(8, truncate=False)
diferencia_rmse = metricas_referencia["rmse"] - metricas_ganador["rmse"]
print(f"MAE del ganador en prueba: {metricas_ganador['mae']:.3f} {UNIDAD}")
if diferencia_rmse > 0:
    print(f"El ganador reduce el RMSE de persistencia en {diferencia_rmse:.3f} {UNIDAD}.")
else:
    print("El ganador no reduce el RMSE de persistencia en la prueba reservada.")

+------------+------------------+------------------+------------------+
|modelo      |rmse              |mae               |r2                |
+------------+------------------+------------------+------------------+
|Elastic Net |1.5436369669794805|1.1603409420814201|0.6198653221651907|
|Persistencia|1.6338743692663322|1.2258741258741264|0.5741227225874275|
+------------+------------------+------------------+------------------+

+-------------------+-------------------+-------------------+-----------------+--------------------+
|inicio_utc         |inicio_objetivo_utc|volumen_promedio_t1|prediction       |error_absoluto      |
+-------------------+-------------------+-------------------+-----------------+--------------------+
|2026-01-05 00:00:00|2026-01-05 00:10:00|8.0                |9.361571286338155|1.3615712863381546  |
|2026-01-05 00:10:00|2026-01-05 00:20:00|8.8                |7.470229046233517|1.3297709537664835  |
|2026-01-05 00:20:00|2026-01-05 00:30:00|10.8               |8

## 7. Guardado y verificación

Se guarda el ganador de validación como `PipelineModel`, con la imputación, el vector de entrada y el estimador ajustados. Después se recarga y se compara la predicción de cada fila de prueba.

In [44]:
RUTA_MODELO.parent.mkdir(parents=True, exist_ok=True)
modelo_ganador.write().overwrite().save(str(RUTA_MODELO))
recargado = PipelineModel.load(str(RUTA_MODELO))

predicciones_recargadas = recargado.transform(df_test).select(
    "inicio_utc", F.col("prediction").alias("prediccion_recargada")
)
comparacion = predicciones_prueba.join(predicciones_recargadas, "inicio_utc")
control = comparacion.agg(
    F.count("*").alias("filas"),
    F.max(F.abs(F.col("prediction") - F.col("prediccion_recargada"))).alias("diferencia_maxima"),
).first()
assert control.filas == df_test.count()
assert control.diferencia_maxima < 1e-10
print(f"Modelo guardado: {nombre_ganador}")
print(f"Ruta: {RUTA_MODELO}")
print(f"Predicciones verificadas al recargar: {control.filas}")

Modelo guardado: Elastic Net
Ruta: /opt/s04-ml-distribuido-regresion/artifacts/leonardo/modelo_ganador
Predicciones verificadas al recargar: 143


## 8. Reflexión para la defensa

Comparar candidatos evita decidir por lo bien que un modelo se ajusta al pasado. El RMSE hace visibles los errores grandes y el MAE expresa el tamaño medio del error en la unidad del indicador. R² aporta una comparación frente a una predicción constante basada en el promedio. La prueba reservada permite ver cuánto cambia el resultado al pasar a un periodo posterior. Si un candidato más complejo aporta poca mejora, también importa el costo de mantenerlo.

**Hallazgo del bloque de ejemplo.** Elastic Net obtuvo el menor RMSE en validación. En las 143 ventanas reservadas, su MAE fue de 1.160 aeronaves y redujo el RMSE de persistencia en 0.090 aeronaves. La mejora observada es pequeña y corresponde al bloque de ejemplo.

Para el informe individual, registra la celda de la etiqueta, las fechas de cada conjunto, la tabla de candidatos, la importancia del bosque, el resultado de prueba y la verificación del modelo recargado. La guía pide reloj y perfil visibles en las capturas.

In [45]:
spark.stop()